# Route C ohne Kompression — der volle HEOM-Propagator auf einem durchgängigen Gitter

## Worum es geht

Die komprimierte Route C nimmt den HEOM-Propagator, eicht ihn mit der Lyapunov-Metrik $W$
und komprimiert ihn dann mit Arnoldi auf $m$ Dimensionen. Das funktioniert, hat aber zwei
Schritte, die man sich genauer ansehen muss:

* Die **Arnoldi-Iteration** braucht $m$ Anwendungen von $\tilde P$ auf einen Vektor. Der
  Krylov-Raum $\mathcal{K}_m(\tilde P, \tilde{\vec\Sigma}_0) = \mathrm{span}\{\tilde{\vec\Sigma}_0, \tilde P\tilde{\vec\Sigma}_0,\dots\}$
  enthält genau die ersten $m$ Zeitschritte. Das ist zwar keine Zeitpropagation im
  physikalischen Sinn — eine Basis ist keine Trajektorie —, aber es ist $m$-mal
  „die Dynamik anfassen".
* Die **Lyapunov-Eichung** löst $(\mathcal{L}-\delta)^\dagger W + W(\mathcal{L}-\delta) = -\mathbb{1}$.
  Das ist ein **dichter** $O(n^3)$-Schritt, und $W^{\pm1/2}$ sind dichte Matrizen.

Dieses Kapitel lässt **beides** weg. Der Schaltkreis arbeitet direkt im vollen HEOM-Raum,
und als Eichung dient allein die **diagonale** ADO-Skalierung. Was dabei herauskommt:

$$\boxed{\;\Vert P\Vert_2 = 10{,}66 \;\xrightarrow{\;D^{-1}\cdot D\;}\; \Vert \tilde P\Vert_2 = 1{,}00068 \qquad\Longrightarrow\qquad p_{\mathrm{total}}(100):\;\;2{,}8\cdot10^{-206}\;\longrightarrow\;0{,}873\;}$$

und die Abweichung vom exakten qutip-`HEOMSolver` beträgt $1{,}9\cdot10^{-9}$ — bei
**null** klassisch propagierten Zeitschritten.

Der Preis ist die Registergröße: statt $m = 128$ (8 Qubits) arbeitet man mit
$\mathcal{D}_{\mathrm{tot}} = 720$ (11 Qubits). Dafür wird die Ablesung erheblich
einfacher, und genau das ist der rote Faden dieses Kapitels: **ohne Kompression ist die
Rücktransformation eine Diagonale statt einer dichten Matrix**, und deshalb braucht die
Diagonale der Dichtematrix am Ende gar keine Basisdrehung mehr.

Alle Zahlen beziehen sich auf das 4-Site-FMO mit Hierarchietiefe 2 und $N_k = 1$, also
45 ADOs und $\mathcal{D}_{\mathrm{tot}} = 45\cdot d^2 = 720$. Die Implementierung liegt in
`heom_full.py`, die Rechnungen in `main.ipynb`.

## Der HEOM-Propagator $P$ bildet eine Halbgruppe

Im nicht-Markovschen Fall ist das Bad kein passiver Zuschauer: Es merkt sich
Wechselwirkungen und tauscht Energie sowie Korrelationen mit dem System aus. Die
Standard-HEOM löst dieses Problem, indem sie den Zustandsraum erweitert. Wir fassen die
vektorisierte Systemdichtematrix $\mathrm{vec}(\rho_S)$ und das dynamische Gedächtnis des
Bades in einem einzigen großen Super-Vektor $\vec\Sigma(t)$ zusammen:

$$\vec{\Sigma}(t) = \begin{pmatrix} \text{vec}(\rho_S(t)) \\ \text{vec}(\mathrm{ADO}_1(t)) \\ \vdots \\ \text{vec}(\mathrm{ADO}_{\mathcal{N}_{\mathrm{ADOs}}-1}(t)) \end{pmatrix} \in \mathbb{C}^{\mathcal{D}_{\mathrm{tot}}}$$

* **$\mathrm{vec}(\rho_S)$ (oberster Block):** Die vektorisierte Systemdichtematrix der
  Dimension $d^2$ (*zeroth-order ADO*).
* **$\mathrm{vec}(\mathrm{ADO}_k)$ (untere Blöcke):** Die vektorisierten Hilfsoperatoren,
  die die quantenmechanischen Bad-Korrelationen speichern.
* **Gesamtdimension:** $\mathcal{D}_{\mathrm{tot}} = \mathcal{N}_{\mathrm{ADOs}}\cdot d^2$,
  hier $45\cdot16 = 720$.

Auf diesem erweiterten Raum ist die Zeitevolution exakt und lokal in der Zeit (formal
Markovsch) und gehorcht dem linearen Differentialgleichungssystem

$$\frac{\mathrm{d}}{\mathrm{d}t}\vec\Sigma(t) = \mathcal{L}_{\mathrm{HEOM}}\,\vec\Sigma(t) \tag{1}$$

mit dem zeitunabhängigen Superoperator $\mathcal{L}_{\mathrm{HEOM}}\in\mathbb{C}^{\mathcal{D}_{\mathrm{tot}}\times\mathcal{D}_{\mathrm{tot}}}$.
Die Zeitentwicklung über einen festen Schritt $\Delta t$ ist exakt das Matrixexponential:

$$P = \exp(\mathcal{L}_{\mathrm{HEOM}}\Delta t) \quad\Longrightarrow\quad \vec\Sigma(t+\Delta t) = P\,\vec\Sigma(t),\qquad \vec\Sigma(n\Delta t) = P^n\,\vec\Sigma(0),\qquad \vec\Sigma(0) = \begin{pmatrix}\mathrm{vec}(\rho_S(0))\\ \mathbf{0}\end{pmatrix} \tag{2}$$

> **Proposition 1.** Auf dem erweiterten Raum $\mathbb{C}^{\mathcal{D}_{\mathrm{tot}}}$
> bildet die diskrete HEOM-Zeitentwicklung eine Halbgruppe: Der Zustand
> $\vec\Sigma_n = P\,\vec\Sigma_{n-1}$ hängt ausschließlich vom unmittelbaren Vorgänger ab.

**Beweis.** Da $\mathcal{L}_{\mathrm{HEOM}}$ zeitunabhängig ist, gilt für das
Matrixexponential die Halbgruppeneigenschaft $\mathrm{e}^{\mathcal{L}(t+s)} = \mathrm{e}^{\mathcal{L}t}\mathrm{e}^{\mathcal{L}s}$,
also $P^{n+m} = P^nP^m$. Für diskrete Zeitschritte folgt unmittelbar

$$\vec\Sigma(n\Delta t) = \mathrm{e}^{\mathcal{L}n\Delta t}\vec\Sigma(0) = \mathrm{e}^{\mathcal{L}\Delta t}\,\mathrm{e}^{\mathcal{L}(n-1)\Delta t}\vec\Sigma(0) = \mathrm{e}^{\mathcal{L}\Delta t}\,\vec\Sigma\big((n-1)\Delta t\big) = P\,\vec\Sigma\big((n-1)\Delta t\big).\qquad\blacksquare$$

Das ist eine reine Ein-Schritt-Rekursion: Zur Bestimmung von $\vec\Sigma_n$ genügt allein
$\vec\Sigma_{n-1}$; frühere Zeitschritte werden nicht gebraucht. Auf dem Gesamtraum
(System + Bad) entsteht per Konstruktion **kein Gedächtnis-Rest**. Genau das ist der
Grund, warum es hier weder ein Gedächtnisfenster $K$ noch klassisch vorberechnete
Historie braucht.

## Das Hindernis: $\Vert P\Vert_2 = 10{,}66$

Die Sz.-Nagy-Dilatation, mit der wir $P$ auf den Schaltkreis bringen, verlangt einen
Skalierungsfaktor $s \ge \Vert P\Vert_2$, und die Erfolgswahrscheinlichkeiten der
Post-Selektion **multiplizieren** sich über die Schritte. Für die Gesamtwahrscheinlichkeit
gilt die Zerlegung

$$p_{\mathrm{total}}(n) = \underbrace{\frac{1}{s^{2n}}}_{\text{Teil 1 (Gatter-Strafe)}}\cdot\underbrace{\frac{\Vert P^n\vec\Sigma_0\Vert^2}{\Vert\vec\Sigma_0\Vert^2}}_{\text{Teil 2 (physikalische Längenänderung)}},\qquad s = \Vert P\Vert_2 \tag{3}$$

Für unser Modell bei $\Delta t = 10\,\mathrm{fs}$ liefert die numerische Auswertung

$$\Vert P\Vert_2 = 10{,}65992 \quad\Longrightarrow\quad \text{Teil 1} = 10{,}66^{-200} \approx 2{,}8\cdot10^{-206}$$

Das ist unbrauchbar. Bemerkenswert ist dabei, **woran** es nicht liegt: der Spektralradius
ist $\rho(P) = 1$, die Dynamik ist also vollkommen brav. Der Ärger kommt allein daher, dass
$P$ stark **nicht-normal** ist — die euklidische Norm sieht ein transientes Wachstum, das
in der Trajektorie gar nicht auftritt. Die Sz.-Nagy-Dilatation berechnet aber stur die
Operatornorm und zahlt für dieses Phantom.

Auch feinere Zeitschritte helfen nicht. Für $\Delta\tau\to0$ gilt
$n\ln\Vert\mathrm{e}^{\mathcal{L}\Delta\tau}\Vert \to T\cdot\omega(\mathcal{L})$ mit der
**numerischen Abszisse**

$$\omega(\mathcal{L}) := \lambda_{\max}\!\left(\tfrac{\mathcal{L}+\mathcal{L}^\dagger}{2}\right)$$

Gemessen ist $\omega(\mathcal{L}) = 1715\,\mathrm{cm}^{-1}$ für den ungeskalierten
Erzeuger — die Strafe hängt also vom Zeithorizont $T$ ab, nicht von der Schrittzahl, und
verschwindet durch feinere Diskretisierung nicht.

Die Auflösung ist dieselbe wie bei der Lyapunov-Route, nur mit einem viel billigeren
Werkzeug: $10{,}66$ ist eine Aussage über das **Koordinatensystem**, nicht über die
Physik. Man muss also die Koordinaten wechseln — und im HEOM-Fall gibt es dafür eine
Wahl, die schon aus der Herleitung der Hierarchie folgt.

## Scaled HEOM — die diagonale Eichung

Die normale HEOM-Formel lautet

$$\frac{d}{dt}\hat{\sigma}_{\mathbf{n}}(t) = \underbrace{ -\frac{i}{\hbar}\mathcal{L}_S \hat{\sigma}_{\mathbf{n}}(t) + \sum_{j,k} \left( \phi_j \theta_{j,0} - n_{jk}\gamma_{jk} \right) \hat{\sigma}_{\mathbf{n}}(t) }_{\text{Term 1: Eigendynamik \& Zerfall}} + \underbrace{ \sum_{j,k} \phi_j \hat{\sigma}_{\mathbf{n}_{jk+}}(t) }_{\text{Term 2: Kopplung nach oben}} + \underbrace{ \sum_{j,k} n_{jk} \theta_{j,k} \hat{\sigma}_{\mathbf{n}_{jk-}}(t) }_{\text{Term 3: Kopplung nach unten}}$$

Wir wollen das in die skalierte Form umwandeln:

$$\frac{d}{dt}\tilde{\sigma}_{\mathbf{n}}(t) = \underbrace{ -\frac{i}{\hbar}\mathcal{L}_S \tilde{\sigma}_{\mathbf{n}} + \sum_{j,k} \left( \phi_j \theta_{j,0} - n_{jk}\gamma_{jk} \right) \tilde{\sigma}_{\mathbf{n}} }_{\text{unveraendert}} + \underbrace{ \sum_{j,k} \sqrt{(n_{jk}+1)\tfrac{|a_{jk}|}{\hbar}} \, \phi_j \tilde{\sigma}_{\mathbf{n}_{jk,+}} }_{\text{scaling\_up}} + \underbrace{ \sum_{j,k} \sqrt{\tfrac{n_{jk}}{|a_{jk}|/\hbar}} \, \theta_{j,k} \tilde{\sigma}_{\mathbf{n}_{jk,-}} }_{\text{scaling\_down}}$$

Dazu definieren wir

$$\tilde{\sigma}_{\mathbf{n}}(t) = \frac{\hat{\sigma}_{\mathbf{n}}(t)}{S_{\mathbf{n}}} \quad \text{mit} \quad S_{\mathbf{n}} = \sqrt{ \prod_{j,k} n_{jk}! \left( \frac{|a_{jk}|}{\hbar} \right)^{n_{jk}} }$$

und rechnen die drei Terme einzeln durch.

### Term 1

$S_{\mathbf n}$ ist eine Zahl, kein Operator, und der Diagonalterm koppelt $\hat\sigma_{\mathbf n}$
nur an sich selbst — der Faktor kürzt sich also vollständig weg:

$$\frac{1}{S_{\mathbf n}} \text{Term 1}(S_{\mathbf{n}} \hat{\sigma}_{\mathbf{n}})= \frac{1}{S_{\mathbf n}} \left( -\frac{i}{\hbar}[H, S_{\mathbf{n}} \hat{\sigma}_{\mathbf{n}}] + \sum_{j,k} \left( \phi_j\theta_{j,0} - n_{jk}\gamma_{jk} \right) S_{\mathbf{n}} \hat{\sigma}_{\mathbf{n}} \right) = \frac{S_{\mathbf{n}}}{S_{\mathbf n}} \left( -\frac{i}{\hbar}[H, \hat{\sigma}_{\mathbf{n}}] + \sum_{j,k} \left( \phi_j\theta_{j,0} - n_{jk}\gamma_{jk} \right) \hat{\sigma}_{\mathbf{n}} \right) = \text{Term 1}(\hat{\sigma}_{\mathbf{n}})$$

### Term 2

Hier koppelt $\mathbf n$ an $\mathbf n_{jk+}$, die Faktoren gehören also zu **verschiedenen**
ADOs und kürzen sich nicht weg. Im Quotienten $S_{\mathbf n_{jk+}}/S_{\mathbf n}$ bleibt
von allen Produkten nur der eine Faktor $(j,k)$ stehen:

$$\begin{align*}
\frac{1}{S_{\mathbf{n}}} \text{Term 2}\left(S_{\mathbf{n}_{jk+}} \hat{\sigma}_{\mathbf{n}_{jk+}}\right) &= \sum_{j,k} \left( \frac{S_{\mathbf{n}_{jk+}}}{S_{\mathbf{n}}} \right) \phi_j \hat{\sigma}_{\mathbf{n}_{jk+}} = \sum_{j,k} \sqrt{ \frac{ (n_{jk}+1)! \left( \frac{|a_{jk}|}{\hbar} \right)^{n_{jk}+1} \prod_{(j',k') \neq (j,k)} n_{j'k'}! \left( \frac{|a_{j'k'}|}{\hbar} \right)^{n_{j'k'}} }{ n_{jk}! \left( \frac{|a_{jk}|}{\hbar} \right)^{n_{jk}} \prod_{(j',k') \neq (j,k)} n_{j'k'}! \left( \frac{|a_{j'k'}|}{\hbar} \right)^{n_{j'k'}} } } \phi_j \hat{\sigma}_{\mathbf{n}_{jk+}} \\
&= \sum_{j,k} \sqrt{ \frac{ (n_{jk}+1) \cdot n_{jk}! \cdot \left( \frac{|a_{jk}|}{\hbar} \right) \cdot \left( \frac{|a_{jk}|}{\hbar} \right)^{n_{jk}} }{ n_{jk}! \cdot \left( \frac{|a_{jk}|}{\hbar} \right)^{n_{jk}} } } \phi_j \hat{\sigma}_{\mathbf{n}_{jk+}} = \sum_{j,k} \sqrt{ (n_{jk}+1) \frac{|a_{jk}|}{\hbar} } \, \phi_j \hat{\sigma}_{\mathbf{n}_{jk+}}
\end{align*}$$

### Term 3

Dasselbe nach unten, wobei der Vorfaktor $n_{jk}$ mitgenommen und unter die Wurzel
gezogen wird:

$$\begin{align*}
\frac{1}{S_{\mathbf{n}}} \text{Term 3}\left(S_{\mathbf{n}_{jk-}} \hat{\sigma}_{\mathbf{n}_{jk-}}\right) &= \sum_{j,k} n_{jk} \left( \frac{S_{\mathbf{n}_{jk-}}}{S_{\mathbf{n}}} \right) \theta_{j,k} \hat{\sigma}_{\mathbf{n}_{jk-}} = \sum_{j,k} n_{jk} \sqrt{ \frac{ (n_{jk}-1)! \left( \frac{|a_{jk}|}{\hbar} \right)^{n_{jk}-1} }{ n_{jk}! \left( \frac{|a_{jk}|}{\hbar} \right)^{n_{jk}} } } \theta_{j,k} \hat{\sigma}_{\mathbf{n}_{jk-}} \\
&= \sum_{j,k} n_{jk} \sqrt{ \frac{ 1 }{ n_{jk} \left( \frac{|a_{jk}|}{\hbar} \right) } } \theta_{j,k} \hat{\sigma}_{\mathbf{n}_{jk-}} = \sum_{j,k} \sqrt{ n_{jk}^2 \cdot \frac{ 1 }{ n_{jk} \left( \frac{|a_{jk}|}{\hbar} \right) } } \theta_{j,k} \hat{\sigma}_{\mathbf{n}_{jk-}} = \sum_{j,k} \sqrt{ \frac{ n_{jk} }{ |a_{jk}| / \hbar } } \theta_{j,k} \hat{\sigma}_{\mathbf{n}_{jk-}}
\end{align*}$$

### Die Intuition dahinter

Mit $\phi_j = i V_j^\times$ und $\theta_{j,k\neq0} = i a_{jk} V_j^\times$, wobei
$a_{jk} = \frac{4\lambda\gamma\nu_k}{\beta(\nu_k^2-\gamma^2)}$, sieht man das Problem
direkt: der Term $\phi_j \hat\sigma_{\mathbf n_{jk+}}$ trägt **keinen** Vorfaktor, der
Term $n_{jk}\theta_{j,k}\hat\sigma_{\mathbf n_{jk-}}$ dagegen den Vorfaktor
$n_{jk}\,a_{jk}/\hbar$. Dieses Ungleichgewicht erzeugt in der Matrix von (1) sehr
verschiedene Eigenwerte und damit ein **steifes** Problem. Selbst kleine $n_{jk}$ pflanzen
sich durch die Hierarchie faktoriell fort. Nach der Skalierung tragen beide Richtungen
denselben Faktor $\sqrt{n_{jk}\,|a_{jk}|/\hbar}$ — die Leiter wird symmetrisch, genau wie
beim harmonischen Oszillator.

Und genau diese Symmetrisierung ist es, die $\Vert P\Vert_2$ rettet.

### Die Skalierung als Ähnlichkeitstransformation

Was oben ADO für ADO ausgerechnet wurde, ist in Matrixform eine **Diagonaltransformation**.
Sei $D = \operatorname{diag}(s_1,\dots,s_{\mathcal{D}_{\mathrm{tot}}})$ die Diagonalmatrix
der Skalenfaktoren, im Code das Array `scale`. Die umskalierte Koordinate ist
$\tilde\Sigma_i = \Sigma_i/s_i$, in Matrixschreibweise

$$\tilde{\vec\Sigma}(t) = D^{-1}\vec\Sigma(t) \qquad\Longleftrightarrow\qquad \vec\Sigma(t) = D\,\tilde{\vec\Sigma}(t) \tag{4}$$

Einsetzen in die Bewegungsgleichung (1) liefert

$$\frac{\mathrm d}{\mathrm dt}\big(D\,\tilde{\vec\Sigma}\big) = \mathcal{L}_{\mathrm{HEOM}}\big(D\,\tilde{\vec\Sigma}\big) \qquad\Longrightarrow\qquad \frac{\mathrm d}{\mathrm dt}\tilde{\vec\Sigma} = \underbrace{\big(D^{-1}\mathcal{L}_{\mathrm{HEOM}}D\big)}_{=:\;\tilde{\mathcal{L}}_{\mathrm{HEOM}}}\tilde{\vec\Sigma}$$

Da $D$ diagonal ist, gilt $D_{jj} = s_j$ und $(D^{-1})_{ii} = 1/s_i$, für das Matrixelement
$(i,j)$ also

$$\big(\tilde{\mathcal{L}}_{\mathrm{HEOM}}\big)_{ij} = (D^{-1})_{ii}\,(\mathcal{L}_{\mathrm{HEOM}})_{ij}\,D_{jj} = (\mathcal{L}_{\mathrm{HEOM}})_{ij}\cdot\frac{s_j}{s_i} \tag{5}$$

Der Ausdruck `scale[None, :] / scale[:, None]` ist genau das äußere Produkt mit den
Einträgen $s_j/s_i$ an Position $(i,j)$; im Code steht deshalb schlicht
`A = A * (scale[None, :] / scale[:, None])`.

> **Proposition 2.** Die ADO-Skalierung ist eine Ähnlichkeitstransformation und damit
> **exakt**: mit $\tilde P := D^{-1}PD$ gilt
> $$\tilde P^{\,n} = D^{-1}P^{\,n}D,\qquad \tilde{\vec\Sigma}_n = \tilde P^{\,n}\tilde{\vec\Sigma}_0 = D^{-1}\vec\Sigma_n$$
> Die Eichung führt **keinerlei** Näherung ein, und sie kostet klassisch $O(n)$ statt
> $O(n^3)$, weil $D$ diagonal und eintragsweise berechenbar ist.

**Beweis.** Die Potenz ist ein Teleskopprodukt, in dem sich $DD^{-1}=\mathbb{1}$ immer
wieder aufhebt:

$$\tilde P^{\,n} = \big(D^{-1}PD\big)^n = D^{-1}P\,\underbrace{DD^{-1}}_{=\,\mathbb{1}}\,P\,\underbrace{DD^{-1}}_{=\,\mathbb{1}}\,P\cdots P\,D = D^{-1}P^{\,n}D$$

Startet man bei $\tilde{\vec\Sigma}_0 = D^{-1}\vec\Sigma_0$, so folgt

$$\tilde{\vec\Sigma}_n = \tilde P^{\,n}\tilde{\vec\Sigma}_0 = \big(D^{-1}P^{\,n}D\big)\big(D^{-1}\vec\Sigma_0\big) = D^{-1}\big(P^{\,n}\vec\Sigma_0\big) = D^{-1}\vec\Sigma_n$$

Multiplikation von links mit $D$ gibt den exakten physikalischen Zustand zurück. Zum
Aufwand: $S_{\mathbf n}$ folgt aus dem Label $\mathbf n$ durch ein Produkt über die
Badexponenten, also $O(K)$ pro ADO und $O(n)$ insgesamt — ohne jede Matrixoperation.
$\;\blacksquare$

Das ist der ganze Unterschied zur Lyapunov-Route. Dort ist $W$ die Lösung einer
Lyapunov-Gleichung, dicht, $O(n^3)$, und $W^{\pm1/2}$ sind dichte Matrizen. Hier ist $D$
eine Diagonale, die man aus dem ADO-Label ausrechnet.

### Reicht die Diagonale? — die Messung

Die Frage ist nicht, ob die Skalierung exakt ist (Proposition 2), sondern ob sie
$\Vert P\Vert_2$ weit genug drückt. Gemessen für das 4-Site-FMO, Tiefe 2,
$\Delta t = 10\,\mathrm{fs}$:

| Eichung | $\Vert P\Vert_2$ | $p_{\mathrm{total}}(100)$ aus der Dilatation | klassische Kosten |
| :--- | ---: | ---: | :--- |
| keine | $10{,}65992$ | $2{,}8\cdot10^{-206}$ | — |
| **ADO-Skalierung (diagonal)** | $\mathbf{1{,}00068}$ | $\mathbf{0{,}873}$ | $O(n)$, eintragsweise |
| ADO + Lyapunov | $1{,}00001$ | $0{,}998$ | $O(n^3)$, dicht |

Die diagonale Skalierung allein bringt $\Vert P\Vert_2$ von $10{,}66$ auf $1{,}00068$ und
damit $p_{\mathrm{total}}(100)$ von $10^{-206}$ auf $0{,}873$. Die Lyapunov-Eichung gewinnt
danach noch $0{,}873 \to 0{,}998$ — für einen dichten $O(n^3)$-Schritt ein schlechtes
Geschäft, und für ein großes System gar keine Option.

Dasselbe an der numerischen Abszisse, die ja die eigentliche Ursache ist:

$$\omega(\mathcal{L}) = 1715{,}16\;\mathrm{cm}^{-1} \qquad\xrightarrow{\;D^{-1}\cdot D\;}\qquad \omega(\tilde{\mathcal{L}}) = 0{,}3614\;\mathrm{cm}^{-1}$$

Vier Größenordnungen, allein durch die Symmetrisierung der Leiteroperatoren. Die
Nicht-Normalität von $P$ war also fast vollständig ein **Koordinatenartefakt** der
üblichen HEOM-Konvention — qutip 5 skaliert die ADOs nicht (das `renorm`-Argument gab es
bis QuTiP 4.6 und ist ersatzlos entfallen), und genau diese Spreizung landet sonst in
$\Vert P\Vert_2$.

Ab hier arbeiten wir also ausschließlich mit $\tilde P = D^{-1}PD$ und schreiben der
Kürze halber wieder $P$ dafür.

## Das Gitter

Die $\mathcal{D}_{\mathrm{tot}}\times\mathcal{D}_{\mathrm{tot}}$-Matrix $\tilde P$ wird auf
$n_p = 2^{\lceil\log_2 \mathcal{D}_{\mathrm{tot}}\rceil}$ aufgefüllt (hier $720\to1024$) und
**einmal** dilatiert:

$$U = \begin{pmatrix} \tilde P/s & B\\ C & -(\tilde P/s)^\dagger\end{pmatrix},\qquad B = \sqrt{\mathbb{1}-\tfrac{\tilde P\tilde P^\dagger}{s^2}},\qquad C = \sqrt{\mathbb{1}-\tfrac{\tilde P^\dagger\tilde P}{s^2}},\qquad s = \Vert\tilde P\Vert_2$$

Der Rest des Raums bleibt null und wird nie besetzt, weil der Anfangszustand im
$720$-dimensionalen Block liegt und $\tilde P$ ihn dort hält. Danach besteht der gesamte
Schaltkreis aus genau einem Gatter, das man wiederholt:

```
heom (10 Qubits) : ────[ U ]─────────────[ U ]─────────────[ U ]────────── ... ───> X_n
                        │                 │                 │
Ancilla          : |0>──[ U ]─(M)──|0>────[ U ]─(M)──|0>────[ U ]─(M)─|0>─ ...
                              │                 │                 │
klass. Register  : ───────────●─────────────────●─────────────────●─────── ...
                            rec[1]            rec[2]            rec[3]
```

**Erstens brauchen wir keine Zustandspräparation.** Das ist ohne Kompression sogar
einfacher zu sehen als mit:

> **Proposition 3.** Für $\rho_S(0) = \vert1\rangle\langle1\vert$ gilt
> $$\tilde{\vec\Sigma}_0 = e_0,\qquad \Vert\tilde{\vec\Sigma}_0\Vert = 1$$
> Das Register startet also ohnehin in $\vert0\dots0\rangle$, und in der Skala
> $\lambda_t$ (siehe unten) fällt der Faktor $\Vert\tilde{\vec\Sigma}_0\Vert$ weg.

**Beweis.** Nach (2) ist $\vec\Sigma_0 = (\mathrm{vec}\rho_S(0),\mathbf 0)^{\mathsf T}$, alle
höheren ADOs starten bei null. In spaltenweiser vec-Konvention ist
$\mathrm{vec}(\vert1\rangle\langle1\vert)_i = \delta_{i0}$, also $\vec\Sigma_0 = e_0$. Der
nullte ADO trägt das Label $\mathbf n = (0,\dots,0)$ und damit nach Definition den
Skalenfaktor

$$S_{(0,\dots,0)} = \sqrt{\prod_{j,k} 0!\left(\tfrac{|a_{jk}|}{\hbar}\right)^{0}} = \sqrt{1} = 1$$

Also ist $s_i = 1$ für alle $i < d^2$, und $\tilde{\vec\Sigma}_0 = D^{-1}e_0 = e_0$ mit
Norm 1. $\;\blacksquare$

Im Code steht dafür die Zusicherung `assert np.allclose(scale[:D], 1.0)` — bricht sie, ist
die Annahme verletzt und der Rest der Rechnung wäre still falsch.

**Zweitens messen wir die Ancilla sofort nach jedem Schritt und setzen sie zurück**, statt
für jeden Schritt ein frisches Ancilla-Qubit zu spendieren. Nach dem Prinzip der
aufgeschobenen Messung ist das exakt äquivalent: Sobald Schritt $t$ vorbei ist, berührt kein
Gatter dieses Ancilla-Qubit je wieder, die Messung kommutiert also mit allem Folgenden. So
kommt man mit **einem** Ancilla-Qubit für beliebig viele Schritte aus; das Messprotokoll
wandert in das klassische Register `rec`.

### Der akzeptierte Zweig ist deterministisch

> **Proposition 4.** Bedingt auf das Messprotokoll $\mathtt{rec} = 0\cdots0$ ist der Zustand
> des Systemregisters nach $t$ Schritten
> $$\vert X_t\rangle = \frac{\tilde P^{\,t}\,\tilde{\vec\Sigma}_0}{\Vert\tilde P^{\,t}\,\tilde{\vec\Sigma}_0\Vert} \tag{6}$$
> und damit **unabhängig von jedem Zufall in den Messungen**.

**Beweis.** Induktion über $t$. Für $t=0$ ist nichts zu zeigen. Sei das Register vor
Schritt $t$ im reinen Produktzustand $\vert X_{t-1}\rangle\otimes\vert0\rangle_E$. Die
Anwendung von $U$ spaltet ihn in genau zwei Zweige auf:

$$U\big(\vert0\rangle_E\otimes\vert X_{t-1}\rangle\big) = \vert0\rangle_E\otimes\Big(\tfrac{\tilde P}{s}X_{t-1}\Big) \;+\; \vert1\rangle_E\otimes\big(C\,X_{t-1}\big)$$

Die Messung wählt zufällig einen der beiden Zweige — welchen, das ist die einzige
Zufälligkeit im ganzen Ablauf. *Gegeben* das Ergebnis $\vert0\rangle$ projiziert der
Kollaps auf den ersten Summanden, und die Normierung hebt den Faktor $1/s$ wieder auf:

$$\vert X_t\rangle = \frac{\big(\tilde P/s\big)X_{t-1}}{\big\Vert\big(\tilde P/s\big)X_{t-1}\big\Vert} = \frac{\tfrac1s\,\tilde PX_{t-1}}{\tfrac1s\,\Vert\tilde PX_{t-1}\Vert} = \frac{\tilde P\,X_{t-1}}{\Vert\tilde P\,X_{t-1}\Vert}$$

Das ist eine **Funktion von $X_{t-1}$ allein**, ohne jeden Rest an Zufall. Das
anschließende `reset` bringt die Ancilla wieder nach $\vert0\rangle$, so dass die
Induktionsvoraussetzung für Schritt $t+1$ wiederhergestellt ist. Setzt man die Rekursion
$t$-mal ein und benutzt $\tilde{\vec\Sigma}_0 = e_0$ aus Proposition 3, folgt (6).
$\;\blacksquare$

Das hat eine sehr praktische Konsequenz. Der Zufall im Schaltkreis steckt *ausschließlich*
in der Frage, **ob** ein Durchlauf akzeptiert wird, nicht darin, **was** man im Erfolgsfall
vorfindet. Ein einziger akzeptierter Shot trägt daher bereits den exakten Zustand; die
Statistik über viele Shots wird nur für die skalare Zahl $p_{\mathrm{total}}$ gebraucht.

### Die Rückrechnung auf die Physik

Der Quantencomputer gibt uns den **normierten** Vektor $X_t$. Um daraus $\rho_S(t)$ zu
machen, brauchen wir zwei Dinge: seine Länge und die Rücktransformation aus den
Eichkoordinaten.

**Die Länge.** Die Post-Selektion multipliziert über die Schritte, und aus (3) mit
$\tilde P$ statt $P$ folgt

$$p_{\mathrm{total}}(t) = \frac{\Vert\tilde P^{\,t}\tilde{\vec\Sigma}_0\Vert^2}{s^{2t}\Vert\tilde{\vec\Sigma}_0\Vert^2} \quad\Longrightarrow\quad \Vert\tilde{\vec\Sigma}_t\Vert = \underbrace{\Vert\tilde{\vec\Sigma}_0\Vert}_{=\,1\;\text{(Prop. 3)}}\cdot\, s^{\,t}\,\sqrt{p_{\mathrm{total}}(t)} \;=:\; \lambda_t \tag{7}$$

**Die Rücktransformation.** Hier liegt der entscheidende Unterschied zur komprimierten
Route. Dort war $R = \big(W^{-1/2}\tilde Q_m\big)_{[1:d^2,:]}$ eine **dichte**
$d^2\times m$-Matrix — das Produkt aus der dichten Lyapunov-Wurzel und der Krylov-Basis.
Hier gibt es weder $W^{-1/2}$ noch $\tilde Q_m$, sondern nur die Diagonale $D$, und die
Projektion $\Pi$ schneidet die ersten $d^2$ Zeilen heraus:

$$\mathrm{vec}\,\rho_S(t\Delta t) = \Pi\,\vec\Sigma_t = \Pi\,D\,\tilde{\vec\Sigma}_t = \underbrace{\operatorname{diag}(s_0,\dots,s_{d^2-1})}_{=\;R,\;\text{diagonal}}\;\tilde{\vec\Sigma}_t\big|_{[0:d^2]} \tag{8}$$

und nach Proposition 3 sind diese $s_i$ sogar alle gleich $1$. Ausgeschrieben, mit der
spaltenweisen vec-Konvention $\mathrm{vec}(\rho)_{i+dj} = \rho_{ij}$:

$$\boxed{\;\rho_{ij}(t) = s_{i+dj}\cdot\tilde\Sigma_{t,\,i+dj}\;} \tag{9}$$

**Ein einziger Eintrag des Zustandsvektors ist ein einziger Eintrag der Dichtematrix.**
Genau das macht die Ablesung im nächsten Abschnitt so viel einfacher als im komprimierten
Fall.

| Größe | Bedeutung | Woher kommt der Wert? | Wann im Ablauf? |
| :--- | :--- | :--- | :--- |
| $\Vert\tilde{\vec\Sigma}_0\Vert$ | Länge des geeichten Startvektors | $= 1$ nach Proposition 3 | gar nicht — fällt weg |
| $s$ | Dilatations-Skalierung | $s = \Vert\tilde P\Vert_2 = 1{,}00068$ | **vor** dem Quanten-Run, rein klassisch |
| $R$ | Rücktransformation in den Systemblock | $R = \operatorname{diag}(\mathtt{scale}[:d^2])$ | **vor** dem Quanten-Run, $O(d^2)$ |
| $p_{\mathrm{total}}(t)$ | kumulierte Erfolgswahrscheinlichkeit | Anteil der Shots mit $\mathtt{rec}[1..t] = 0\cdots0$ | **nach** dem Run, aus dem klassischen Register |
| $X_t$ | normierter Registerzustand | Zustand der $\log_2 n_p$ Qubits, bedingt auf die akzeptierten Shots | am Zeitschritt $t$ |

# Messergebnisse der Diagonalterme aus Zählraten

In der Theorie verwenden wir Gleichung (9), $\rho_{ij}(t) = s_{i+dj}\cdot\tilde\Sigma_{t,i+dj}$,
um den Zustand am Ende des Quantencircuits wieder in die richtige Basis zu transformieren
und die Systemdichtematrix zu erhalten. Wir hatten ja zuvor eine Basistransformation
(Ähnlichkeitstransformation) mit $D$ durchgeführt.

In einer realen Anwendung können wir den Zustand $X_t$ aber nicht so einfach auslesen,
dafür wäre Quantentomographie nötig. Wir können also nur Counts auslesen. Angenommen, ein
System aus $n$ Qubits befindet sich vor der Messung in einer quantenmechanischen
Überlagerung

$$\vert\psi\rangle = c_0\vert00\dots0\rangle + c_1\vert00\dots1\rangle + \dots + c_{2^n-1}\vert11\dots1\rangle = \sum_{k=0}^{2^n-1} c_k\vert k\rangle$$

Führt man einen einzelnen Shot aus:

- Das Quantensystem kollabiert instantan auf genau einen Basiszustand $\vert k\rangle$.
- Die Hardware gibt einen klassischen Bitstring aus (z. B. `'0101010101'`).
- Man erhält nur ein einziges Bitmuster, keine Information über die Amplituden $c_k$.

Um die Amplituden zu rekonstruieren, wiederholt man das gesamte Experiment
(Präparation $\to$ Gatter $\to$ Messung) $N_{\text{shots}}$ Mal. Dabei führt man eine
Strichliste und zählt, wie oft jeder Bitstring $k$ gemessen wurde. Nach dem Gesetz der
großen Zahlen nähert sich die relative Häufigkeit für große Shot-Zahlen exakt der
quantenmechanischen Wahrscheinlichkeit an:

$$P(k) = \lim_{N_{\text{shots}}\to\infty}\frac{N_k}{N_{\text{shots}}} = \vert c_k\vert^2$$

Wir erhalten also nur Wahrscheinlichkeiten und keinen vollen Vektor aus dem Messprozess.

## Warum es hier **kein** Gatter $B_j$ braucht

Im komprimierten Fall war das ein Problem: dort ist die Ablesezeile $r_j$ eine **dichte**
Zeile über den ganzen Krylov-Raum, das Zielprodukt $\rho_{jj} = r_j\cdot y_t$ also eine
Linearkombination aller Komponenten. Aus Betragsquadraten allein bekommt man die nicht,
und deshalb musste man ein Zusatzgatter $B_j$ mit $B_j\vert0\dots0\rangle = \vert\chi_j\rangle$
anhängen, das die gewünschte Richtung erst in die Rechenbasis dreht.

Ohne Kompression entfällt dieser ganze Schritt. Nach (9) ist

$$r_{jj} = s_{j+dj}\cdot e_{j+dj}^{\mathsf T}$$

also ein **skalierter Basisvektor**. Der zugehörige Analysezustand ist damit

$$\vert\chi_{jj}\rangle = \frac{r_{jj}^\dagger}{\Vert r_{jj}\Vert_2} = \frac{s_{j+dj}\,e_{j+dj}}{s_{j+dj}} = \vert j+dj\rangle$$

Das ist bereits ein Zustand der Rechenbasis. Das Gatter, das ihn dorthin drehen müsste, ist
also die Identität: $B_{jj} = \mathbb{1}$. **Man misst einfach das Systemregister in der
Rechenbasis**, und das Histogramm enthält alle $d$ Populationen gleichzeitig — statt $d$
getrennter Schaltkreise wie im komprimierten Fall.

Der Preis dieser Vereinfachung steht in der Registergröße: 11 Qubits statt 8. Man handelt
Gatterbreite gegen Ablesekomfort.

## Die Herleitung der Formel

Wir bauen die Formel jetzt Schritt für Schritt aus dem zusammen, was der Schaltkreis
tatsächlich liefert.

**Schritt 1: Was der Schaltkreis herausgibt.** Nach Proposition 4 ist der Zustand des
Systemregisters, bedingt auf $\mathtt{rec} = 0\cdots0$, der **normierte** Vektor

$$\vert X_t\rangle = \frac{\tilde{\vec\Sigma}_t}{\Vert\tilde{\vec\Sigma}_t\Vert} = \frac{\tilde{\vec\Sigma}_t}{\lambda_t} \qquad\Longleftrightarrow\qquad \tilde\Sigma_{t,i} = \lambda_t\cdot X_{t,i} \tag{10}$$

mit $\lambda_t$ aus (7). Die Normierung nimmt die Natur vor — die Information über die
Länge steckt nicht mehr im Zustand, sondern im Messprotokoll.

**Schritt 2: Was die Messung herausgibt.** Misst man das Systemregister in der Rechenbasis,
liefert die Bornsche Regel für den Basiszustand $\vert k\rangle$

$$q_k(t) := P\big(\vert k\rangle \;\big\vert\; \mathtt{rec}=0\cdots0\big) = \big\vert\langle k\vert X_t\rangle\big\vert^2 = \vert X_{t,k}\vert^2 \qquad\Longrightarrow\qquad \vert X_{t,k}\vert = \sqrt{q_k(t)} \tag{11}$$

Die Bedingung auf $\mathtt{rec}=0\cdots0$ ist wichtig: gezählt wird nur unter den Shots, die
die Post-Selektion überlebt haben.

**Schritt 3: Welcher Basiszustand gehört zu $\rho_{jj}$?** Nach der spaltenweisen
vec-Konvention sitzt $\rho_{jj}$ an der Stelle

$$k_j := j + d\cdot j$$

Im Code ist das genau `_vec_index(d, j, j)`. Für $d=4$ sind das die Indizes
$0, 5, 10, 15$ — die Diagonale der $4\times4$-Matrix, spaltenweise durchnummeriert.

**Schritt 4: Zusammensetzen.** Wir setzen (10) in (9) ein und benutzen (11):

$$\rho_{jj}(t) \;\overset{(9)}{=}\; s_{k_j}\cdot\tilde\Sigma_{t,k_j} \;\overset{(10)}{=}\; s_{k_j}\cdot\lambda_t\cdot X_{t,k_j} \;\overset{(11)}{=}\; s_{k_j}\cdot\lambda_t\cdot\sqrt{q_{k_j}(t)} \tag{12}$$

**Schritt 5: Warum die Wurzel eindeutig ist.** Im letzten Schritt haben wir $X_{t,k_j}$
durch $\vert X_{t,k_j}\vert = \sqrt{q_{k_j}}$ ersetzt, das Vorzeichen und die Phase also
weggeworfen. Das ist hier erlaubt, und zwar aus zwei Gründen zusammen:

* $\rho_{jj}$ ist eine **Population**, also reell und $\ge0$ — das ist eine Eigenschaft
  jeder Dichtematrix, keine Zusatzannahme.
* $s_{k_j} > 0$ nach Definition der Skalenfaktoren (Produkt aus Fakultäten und Beträgen).

Also ist auch $\tilde\Sigma_{t,k_j} = \rho_{jj}/s_{k_j} \ge 0$ und damit
$X_{t,k_j} = \vert X_{t,k_j}\vert$. Die globale Phase, die ein normierter Quantenzustand
ohnehin nur bis auf $\mathrm{e}^{i\varphi}$ festlegt, fällt im Betragsquadrat heraus und
wird durch die Forderung $\rho_{jj}\ge0$ eindeutig fixiert.

**Das Endergebnis.** Setzt man $\lambda_t$ aus (7) ein und benutzt
$\Vert\tilde{\vec\Sigma}_0\Vert = 1$:

$$\boxed{\;\rho_{jj}(t) = \underbrace{s_{j+dj}}_{\text{Skalenfaktor}}\;\cdot\;\underbrace{s^{\,t}\,\sqrt{p_{\mathrm{total}}(t)}}_{=\;\lambda_t}\;\cdot\;\underbrace{\sqrt{q_{j+dj}(t)}}_{\text{aus dem Histogramm}}\;} \tag{13}$$

- $p_{\text{total}}(t)$ ist der Anteil aller Shots, die bei den Ancilla-Messungen
  durchgängig Nullen hatten.
- $q_{j+dj}(t)$ ist der Anteil **dieser überlebenden** Shots, deren Systemregister den
  Bitstring von $j+dj$ zeigt.

## Formel und Code nebeneinander

Genau das steht in `heom_full.run_full_counts`. Der Code lautet

```python
prob[k] = acc / shots                                       # p_total(t)
lam     = s ** t * np.sqrt(prob[k])                         # lambda_t
pop = np.array([scale[_vec_index(d, j, j)] * lam
                * np.sqrt(hist.get(_vec_index(d, j, j), 0) / acc)
                for j in range(d)])
```

und Zeile für Zeile:

| Code | Formel | woher |
| :--- | :--- | :--- |
| `acc` | $N_{\mathrm{akz}}$ — Shots mit $\mathtt{rec}=0\cdots0$ | `_split_counts`, Post-Selektion |
| `shots` | $N_{\mathrm{shots}}$ | Parameter |
| `prob[k] = acc / shots` | $p_{\mathrm{total}}(t)$ | Definition der Post-Selektion |
| `s ** t` | $s^{\,t}$ | Dilatations-Skalierung, $s = \Vert\tilde P\Vert_2$ |
| `lam` | $\lambda_t = s^t\sqrt{p_{\mathrm{total}}}$ | Gleichung (7), mit $\Vert\tilde{\vec\Sigma}_0\Vert=1$ |
| `_vec_index(d, j, j)` | $k_j = j + d\,j$ | spaltenweise vec-Konvention |
| `scale[_vec_index(d,j,j)]` | $s_{k_j}$ | Diagonale von $D$, hier $=1$ |
| `hist.get(..., 0)` | $N_{k_j}$ — Treffer auf Basiszustand $k_j$ | Histogramm der akzeptierten Shots |
| `hist.get(...) / acc` | $q_{k_j}(t)$ | Gleichung (11) |
| `np.sqrt(...)` | $\sqrt{q_{k_j}}$ | Gleichung (12), Wurzel eindeutig nach Schritt 5 |
| die ganze Zeile | Gleichung (13) | |

Das `hist.get(idx, 0)` mit Vorgabewert $0$ ist kein Schönheitsfehler, sondern die
Nachweisgrenze: Kommt der Bitstring $k_j$ in keinem einzigen akzeptierten Shot vor, ist
$q_{k_j}$ nicht klein, sondern **unbestimmt** — man liest $\rho_{jj}=0$ ab und muss wissen,
dass das eine obere Schranke ist, kein Messwert. Wann das passiert, klärt der nächste
Abschnitt.

## Die Fehlerfortpflanzung und die Zahl der benötigten Shots

Das Ziel ist, vor dem Lauf auf einem Quantencomputer mit einer klassischen Rechnung die
nötige Shot-Zahl abzuschätzen. Da man dafür die Dynamik klassisch kennen muss, steht dieser
Weg einer echten Anwendung nicht zur Verfügung — wohl aber der Planung.

Aus (13) folgt: $\rho_{jj}$ hängt von den Zählraten nur über $\sqrt{q}$ und
$\sqrt{p_{\mathrm{total}}}$ ab. Beide Größen sind Binomialschätzer, und für einen
Binomialanteil $q$ aus $N$ Versuchen gilt $\delta q = \sqrt{q(1-q)/N} \approx \sqrt{q/N}$
für $q\ll1$. Die Ableitung der Wurzel liefert $\mathrm{d}\sqrt q/\mathrm{d}q = 1/(2\sqrt q)$,
also

$$\delta\rho_{jj} = s_{k_j}\,\lambda_t\cdot\frac{\partial\sqrt q}{\partial q}\,\delta q = s_{k_j}\,\lambda_t\cdot\frac{1}{2\sqrt q}\sqrt{\frac{q}{N_{\mathrm{akz}}}} = \frac{s_{k_j}\,\lambda_t}{2\sqrt{N_{\mathrm{akz}}}} \tag{14}$$

Das $\sqrt q$ aus der Statistik kürzt sich gegen das $1/\sqrt q$ aus der Ableitung — der
**absolute** Fehler ist von $q$ **unabhängig**. Im Code steht deshalb genau

```python
dpop = np.array([scale[_vec_index(d, j, j)] * lam / (2 * np.sqrt(acc))
                 for j in range(d)])
```

Und es kürzt sich noch mehr. Mit $\lambda_t = s^t\sqrt{p_t}$ und
$N_{\mathrm{akz}} = N_{\mathrm{shots}}\,p_t$ wird aus (14)

$$\delta\rho_{jj} = \frac{s_{k_j}\,s^{t}\sqrt{p_t}}{2\sqrt{N_{\mathrm{shots}}\,p_t}} = \frac{s_{k_j}\,s^{\,t}}{2\sqrt{N_{\mathrm{shots}}}} \tag{15}$$

**Die Post-Selektion kostet für diese Ablesung keinen einzigen zusätzlichen Shot.** Das ist
kein Zufall: verwirft man Shots, wird $\lambda_t$ kleiner, also $q$ größer, und der relative
Fehler auf $q$ genau um denselben Faktor kleiner. Was der Verwurf an Statistik nimmt, gibt
die Verstärkung zurück. Übrig bleibt allein das langsame Wachstum von $s^t$; bei
$s = 1{,}00068$ sind das über 100 Schritte $7\,\%$.

Für absolute Genauigkeit $\eta$ folgt daraus unmittelbar

$$N_{\mathrm{shots}} \;\ge\; \frac{\big(s_{k_j}\,s^{\,t}\big)^2}{4\,\eta^2} \tag{16}$$

### Die Nachweisgrenze

Für **relative** Genauigkeit sieht es anders aus, und dort taucht $q$ wieder auf:

$$\varepsilon = \frac{\delta\rho_{jj}}{\rho_{jj}} = \frac{s_{k_j}\lambda_t/(2\sqrt{N_{\mathrm{akz}}})}{s_{k_j}\lambda_t\sqrt{q}} = \frac{1}{2\sqrt{N_{\mathrm{shots}}\,p_t\,q}} \qquad\Longrightarrow\qquad N_{\mathrm{shots}} \ge \frac{1}{4\varepsilon^2\,q\,p_t} \tag{17}$$

Eine Population nahe null ist eben **absolut** leicht zu messen und **relativ** beliebig
schwer. Praktisch schlägt das als Nachweisgrenze durch: der erste Treffer auf dem
Basiszustand $k_j$ kommt im Mittel bei

$$N_{\mathrm{shots}} \approx \frac{1}{q_{k_j}\,p_t} \approx \frac{1}{q_{k_j}}$$

Gemessen für unser Modell bei $\Delta t = 10\,\mathrm{fs}$:

| $t$ | $\rho_{33}$ | erster Treffer bei | $\rho_{44}$ | erster Treffer bei |
| ---: | ---: | ---: | ---: | ---: |
| 2 | $0{,}0008$ | $1{,}4\cdot10^{6}$ Shots | $0{,}0005$ | $3{,}4\cdot10^{6}$ Shots |
| 4 | $0{,}0066$ | $2{,}3\cdot10^{4}$ Shots | $0{,}0025$ | $1{,}6\cdot10^{5}$ Shots |
| 6 | $0{,}0188$ | $2{,}8\cdot10^{3}$ Shots | $0{,}0060$ | $2{,}8\cdot10^{4}$ Shots |
| 8 | $0{,}0294$ | $1{,}1\cdot10^{3}$ Shots | $0{,}0103$ | $9{,}2\cdot10^{3}$ Shots |

Bei den üblichen $128$ bis $4096$ Shots liest man dort also exakt $0$ ab. Der Fehlerbalken
aus (15) deckt das ab — er ist ja $q$-unabhängig —, aber informativ ist die Null nicht. Das
ist auch der Grund, warum die gemessene Spur typisch bei $0{,}95$ statt $1{,}00$ liegt: die
beiden kleinsten Populationen fehlen einfach.

# Messergebnisse der Off-Diagonalterme

Für $i \neq j$ hilft der Trick aus dem letzten Abschnitt nicht mehr. Zwar ist $\rho_{ij}$
nach (9) immer noch ein einzelner Eintrag des Zustandsvektors, aber er ist **komplex**, und
ein Betragsquadrat verliert die Phase:

$$q_{i+dj} = \vert X_{t,i+dj}\vert^2 \qquad\Longrightarrow\qquad \vert\rho_{ij}\vert = s_{i+dj}\,\lambda_t\sqrt{q_{i+dj}}$$

Man bekommt also den Betrag, aber weder Real- noch Imaginärteil. Die Lösung ist, statt der
Rechenbasis in **gedrehten** Basen zu messen. Mit
$\vert+\rangle = \tfrac{1}{\sqrt2}(\vert a\rangle+\vert b\rangle)$ und
$\vert R\rangle = \tfrac{1}{\sqrt2}(\vert a\rangle - i\vert b\rangle)$ gilt

$$\rho_{++} := \langle+\vert\rho\vert+\rangle = \tfrac12(\rho_{aa}+\rho_{bb}) + \mathrm{Re}\,\rho_{ab},\qquad \rho_{RR} := \langle R\vert\rho\vert R\rangle = \tfrac12(\rho_{aa}+\rho_{bb}) + \mathrm{Im}\,\rho_{ab} \tag{18}$$

Beides sind **echte Besetzungswahrscheinlichkeiten**, also reell und $\ge0$ — die Wurzel
ist damit wieder eindeutig, genau wie in Schritt 5 oben, und es braucht keine
Phasenreferenz.

> **Proposition 5.** Für jeden Zustand $\rho\succeq0$ und jedes $\theta\in[0,2\pi)$ ist
> $$\rho_\theta := \tfrac12(\rho_{aa}+\rho_{bb}) + \mathrm{Re}\big(\mathrm{e}^{-i\theta}\rho_{ab}\big) \;\ge\; 0$$
> insbesondere also $\rho_{++}\ge0$ ($\theta=0$) und $\rho_{RR}\ge0$ ($\theta=\pi/2$).

**Beweis.** $\rho_\theta = \langle\theta\vert\rho\vert\theta\rangle$ mit
$\vert\theta\rangle = \tfrac{1}{\sqrt2}(\vert a\rangle + \mathrm{e}^{i\theta}\vert b\rangle)$
ist ein Erwartungswert und damit $\ge0$, weil $\rho\succeq0$. Direkt an den Einträgen sieht
man, wie viel Luft bleibt: aus $\rho\succeq0$ folgt für den $2\times2$-Block die
Positivität der Determinante, also $\vert\rho_{ab}\vert^2\le\rho_{aa}\rho_{bb}$, und mit
AM–GM

$$\big\vert\mathrm{Re}\,(\mathrm{e}^{-i\theta}\rho_{ab})\big\vert \le \vert\rho_{ab}\vert \overset{\text{C.–S.}}{\le} \sqrt{\rho_{aa}\rho_{bb}} \overset{\text{AM–GM}}{\le} \tfrac12(\rho_{aa}+\rho_{bb}) \qquad\blacksquare$$

**Die Ablesezeilen.** Aus der Linearität von (9) folgen sie unmittelbar:

$$r_{++} = \tfrac12\big(r_{aa} + r_{bb} + r_{ab} + r_{ba}\big),\qquad r_{RR} = \tfrac12\big(r_{aa} + r_{bb} - i\,r_{ab} + i\,r_{ba}\big) \tag{19}$$

und danach klassisch

$$\mathrm{Re}\,\rho_{ab} = \rho_{++} - \tfrac12(\rho_{aa}+\rho_{bb}),\qquad \mathrm{Im}\,\rho_{ab} = \rho_{RR} - \tfrac12(\rho_{aa}+\rho_{bb}) \tag{20}$$

**Hier braucht es nun doch ein Gatter.** $r_{++}$ und $r_{RR}$ sind keine Basisvektoren mehr,
sondern leben auf **genau vier** Koordinaten: $aa$, $bb$, $ab$, $ba$. Man braucht also die
Drehung $B$ mit $B\vert0\dots0\rangle = \vert\chi\rangle$, $\chi = \overline{r}/\Vert r\Vert$,
und misst danach die Wahrscheinlichkeit von $\vert0\dots0\rangle$ — die ist
$\vert\langle\chi\vert X_t\rangle\vert^2$. Die Konstruktion von $B$ über die QR-Zerlegung
steht unverändert im Anhang der komprimierten Fassung; im Code ist es `_prep_unitary`.

Der Unterschied zum komprimierten Fall bleibt aber bestehen: dort ist $\chi$ über den
**ganzen** Raum verteilt, hier auf vier Koordinaten. Klassisch ist das der Unterschied
zwischen einer dichten Drehung und einem eingebetteten $4\times4$-Block.

**Kosten.** $1 + 2\,\vert\text{Paare}\vert$ Schaltkreise je Auslesezeit — einer für die
ganze Diagonale, zwei je Kohärenz. Im komprimierten Fall sind es $d + 2\,\vert\text{Paare}\vert$.

**Fehler.** Nach (20) ist $\mathrm{Re}\,\rho_{ab}$ eine Summe **unabhängig** gemessener
Größen (getrennte Schaltkreise, getrennte Shots), die Varianzen addieren sich also mit den
Quadraten der Koeffizienten $1, -\tfrac12, -\tfrac12$:

$$\delta\big(\mathrm{Re}\,\rho_{ab}\big) = \sqrt{\delta\rho_{++}^2 + \tfrac14\delta\rho_{aa}^2 + \tfrac14\delta\rho_{bb}^2}$$

also bei gleicher Shot-Zahl je Kanal der Faktor $\sqrt{1+\tfrac14+\tfrac14} = 1{,}22$
gegenüber einer einzelnen Population — und wegen $\delta\propto N^{-1/2}$ Faktor $1{,}5$ in
der Shot-Zahl.

## Zusammenfassung des Kapitels

Der Weg in einem Absatz: Die HEOM-Hierarchie ist auf dem erweiterten Raum bereits eine
Halbgruppe, $\vec\Sigma_n = P^n\vec\Sigma_0$ ist exakt (Proposition 1), und es braucht
weder ein Gedächtnisfenster noch klassisch vorberechnete Historie. Das einzige Hindernis
ist, dass $P$ mit $\rho(P)=1$ zwar brav, mit $\Vert P\Vert_2 = 10{,}66$ aber stark
nicht-normal ist. Die Auflösung ist, dass diese Zahl eine Aussage über das
**Koordinatensystem** ist, und im HEOM-Fall liefert schon die Herleitung der Hierarchie die
richtige Wahl: die Skalierung $\tilde\sigma_{\mathbf n} = \hat\sigma_{\mathbf n}/S_{\mathbf n}$
symmetrisiert die Leiteroperatoren, ist eine **diagonale** Ähnlichkeitstransformation und
damit exakt (Proposition 2). Dadurch startet das Register ohnehin in $\vert0\dots0\rangle$
(Proposition 3), der akzeptierte Zweig ist deterministisch (Proposition 4), und die
Rücktransformation ist eine Diagonale statt einer dichten Matrix — weshalb die Diagonale
der Dichtematrix ohne jede Basisdrehung aus **einem** Histogramm fällt.

$$\boxed{\;\Vert P\Vert_2 = 10{,}66 \;\xrightarrow{\;D^{-1}\cdot D\;}\; \Vert\tilde P\Vert_2 = 1{,}00068 \qquad\Longrightarrow\qquad p_{\mathrm{total}}(100):\;\;2{,}8\cdot10^{-206}\;\longrightarrow\;0{,}873\;}$$

| Proposition | Aussage |
| :---: | :--- |
| **1** | HEOM ist auf dem ADO-Raum eine Halbgruppe; kein Gedächtnisrest, kein $K$ |
| **2** | $\tilde P = D^{-1}PD$ ist ähnlich zu $P$ — die Eichung ist exakt und kostet $O(n)$ |
| **3** | $\tilde{\vec\Sigma}_0 = e_0$ — keine State-Preparation, $\Vert\tilde{\vec\Sigma}_0\Vert = 1$ |
| **4** | der auf $0\cdots0$ bedingte Zweig ist deterministisch — ein Shot genügt für den Zustand |
| **5** | $\rho_\theta \ge 0$ für jeden Winkel — die Wurzel bei der Kohärenzablesung ist eindeutig |

### Was gegenüber der komprimierten Fassung wegfällt

| | komprimiert | **hier** |
| :--- | :--- | :--- |
| Eichung | Lyapunov, $W$ dicht, $O(n^3)$ | **ADO-Skalierung, diagonal, $O(n)$** |
| Arnoldi | $m$ Anwendungen von $\tilde P$ | **entfällt** |
| Rücktransformation $R$ | dicht, $d^2\times m$ | **diagonal, $d^2$ Einträge** |
| Diagonale ablesen | $d$ Schaltkreise mit je einem $B_j$ | **1 Schaltkreis, kein $B$** |
| Analysevektor $\chi$ | über den ganzen Raum verteilt | **auf 4 Koordinaten** |
| Qubits | 8 | 11 |
| Abweichung von qutip | $5{,}3\cdot10^{-7}$ | $1{,}9\cdot10^{-9}$ |

### Was bleibt

Zwei klassische Schritte sind weiterhin **dicht** und $O(n^3)$: $P = \exp(\mathcal{L}\Delta t)$
und die Dilatation mit ihren beiden Matrixwurzeln. Keiner davon hängt von der Zahl der
Zeitschritte ab — das ist die Aussage dieser Arbeit und sie hält. Aber beide hängen sehr
wohl von der Hilbertraum-**Dimension** ab, und das ist eine andere Aussage. Der Grund liegt
in einer einzigen Messung: $\mathcal{L}$ ist **dünn besetzt** (8,2 Nichtnullen je Zeile,
1,15 %), $\exp(\mathcal{L}\Delta t)$ dagegen zu **93,8 % dicht**. Das Matrixexponential
zerstört genau die Struktur, die man für ein großes System bräuchte. Was daraus für die
Skalierbarkeit folgt, steht in `kritik.ipynb` und `skalierbarkeit.ipynb`.

Die Implementierung liegt in `heom_full.py`, die Rechnungen und alle Abbildungen in
`main.ipynb`.